# Supply Chain Data Exploration & Understanding

This notebook loads the synthetic datasets and profiles the supply-chain quality landscape.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from pathlib import Path

plt.style.use('seaborn-v0_8-whitegrid')
sns.set_context('talk')

root = Path.cwd()

suppliers = pd.read_csv(root / 'data' / 'suppliers.csv')
batches = pd.read_csv(root / 'data' / 'component_batches.csv')
qc = pd.read_csv(root / 'data' / 'qc_results.csv')
defects = pd.read_csv(root / 'data' / 'defects.csv')

# Display first rows
for name, df in [('suppliers', suppliers), ('batches', batches), ('qc', qc), ('defects', defects)]:
    print(f'\n=== {name.upper()} ===')
    display(df.head())
    print(df.shape)
    print(df.dtypes.to_string())
    print('Missing values:', df.isna().sum().to_dict())


In [ ]:
# Summary metrics
print('Total batches:', batches['batch_id'].nunique())
print('Total units:', qc['unit_id'].nunique())
print('Total defects:', defects['defect_id'].nunique())
print('Overall defect rate:', round(len(defects) / len(qc), 4) * 100, '%')
print('Mean batch size:', round(batches['quantity_produced'].mean(), 2))
print('Median batch size:', round(batches['quantity_produced'].median(), 2))
print('Std batch size:', round(batches['quantity_produced'].std(), 2))

# Supplier defect rate
qc_failures = qc[qc['result'] == 'Fail'].copy()
fail_count_by_batch = qc_failures.groupby('batch_id').size().reset_index(name='failed_units')
merged = batches.merge(fail_count_by_batch, on='batch_id', how='left')
merged['failed_units'] = merged['failed_units'].fillna(0)
merged['failure_rate'] = merged['failed_units'] / merged['quantity_produced']

supplier_risk = merged.groupby('supplier_id')['failure_rate'].mean().reset_index()
print('Top 5 high-risk suppliers:
', supplier_risk.sort_values('failure_rate', ascending=False).head().to_string(index=False))

component_risk = merged.groupby('component_name')['failure_rate'].mean().reset_index()
print('Component defect rates:
', component_risk.sort_values('failure_rate', ascending=False).to_string(index=False))


In [ ]:
# Histogram: Distribution of batch sizes
plt.figure(figsize=(10, 5))
sns.histplot(batches['quantity_produced'], bins=40, kde=True, color='steelblue')
plt.title('Distribution of Batch Sizes')
plt.xlabel('Quantity Produced')
plt.ylabel('Frequency')
plt.tight_layout()


In [ ]:
# Box plot: Defect rates by supplier
merged['supplier_id'] = merged['supplier_id'].astype(str)
plt.figure(figsize=(12, 6))
sns.boxplot(data=merged, x='supplier_id', y='failure_rate', palette='viridis')
plt.title('Defect Rate Distribution by Supplier')
plt.xlabel('Supplier ID')
plt.ylabel('Failure Rate')
plt.xticks(rotation=90)
plt.tight_layout()


In [ ]:
# Time series: Batches produced per month
batches['production_date'] = pd.to_datetime(batches['production_date'])
monthly_batches = batches.groupby(batches['production_date'].dt.to_period('M')).size().reset_index(name='count')
monthly_batches['production_date'] = monthly_batches['production_date'].astype(str)

plt.figure(figsize=(12, 5))
sns.lineplot(data=monthly_batches, x='production_date', y='count', marker='o', color='darkgreen')
plt.title('Batches Produced per Month')
plt.xlabel('Month')
plt.ylabel('Number of Batches')
plt.xticks(rotation=45)
plt.tight_layout()


In [ ]:
# Bar chart: defect count by defect type (top 10)
defect_counts = defects['defect_type'].value_counts().head(10)
plt.figure(figsize=(10, 6))
sns.barplot(x=defect_counts.values, y=defect_counts.index, palette='magma')
plt.title('Top 10 Defect Types')
plt.xlabel('Count')
plt.ylabel('Defect Type')
plt.tight_layout()


In [ ]:
# Pie chart: distribution of checkpoint failures
checkpoint_distribution = defects['checkpoint_id'].value_counts()
plt.figure(figsize=(8, 8))
plt.pie(checkpoint_distribution.values, labels=[f'Checkpoint {k}' for k in checkpoint_distribution.index], autopct='%1.1f%%', startangle=90)
plt.title('Distribution of Failure Checkpoints')
plt.tight_layout()


In [ ]:
# Scatter: quantity produced vs quantity passed QC
plt.figure(figsize=(8, 6))
sns.scatterplot(data=batches, x='quantity_produced', y='quantity_passed_qc', alpha=0.5, color='crimson')
plt.title('Quantity Produced vs. Quantity Passed QC')
plt.xlabel('Quantity Produced')
plt.ylabel('Quantity Passed QC')
plt.tight_layout()


In [ ]:
# Heatmap: Supplier count by country
heatmap = suppliers.groupby('country')['supplier_id'].count().unstack(fill_value=0)
plt.figure(figsize=(8, 6))
sns.heatmap(heatmap, annot=True, fmt='d', cmap='Blues')
plt.title('Supplier Count by Country')
plt.tight_layout()


In [ ]:
# KDE plot: Distribution of quality scores
plt.figure(figsize=(8, 5))
sns.kdeplot(suppliers['quality_score'], fill=True, color='purple')
plt.title('Distribution of Supplier Quality Scores')
plt.xlabel('Quality Score')
plt.ylabel('Density')
plt.tight_layout()


In [ ]:
# Histogram: Lead time distribution
plt.figure(figsize=(8, 5))
sns.histplot(suppliers['lead_time_days'], bins=20, kde=True, color='darkorange')
plt.title('Lead Time Distribution')
plt.xlabel('Lead Time (days)')
plt.ylabel('Frequency')
plt.tight_layout()


In [ ]:
# Time series: Cumulative defects over 24 months
defects['reported_date'] = pd.to_datetime(defects['reported_date'])
monthly_defects = defects.groupby(defects['reported_date'].dt.to_period('M')).size().reset_index(name='count')
monthly_defects['cumulative'] = monthly_defects['count'].cumsum()
monthly_defects['reported_date'] = monthly_defects['reported_date'].astype(str)

plt.figure(figsize=(12, 5))
sns.lineplot(data=monthly_defects, x='reported_date', y='cumulative', marker='o', color='teal')
plt.title('Cumulative Defects Over 24 Months')
plt.xlabel('Month')
plt.ylabel('Cumulative Defects')
plt.xticks(rotation=45)
plt.tight_layout()
